# 03_concurrency: What Breaks at Scale

[Open in Colab](https://colab.research.google.com/github/Utkarsh-09/AI_GURU_labs/blob/main/notebooks/03_concurrency.ipynb)

**Session:** Day 2, S8 — What breaks at scale (40-minute block: this lab first, then the talk)
**Expected runtime:** **15 minutes**, the tightest budget of the week, so the lab is small: two TODOs, one load test, one chart, one table, three numbers to carry to the sizing worksheet. The machine part is one load test of 80 requests (5 concurrency levels x 16 requests, 64-token replies). Measured on the build laptop (Ollama 0.12.10, `llama3.2:1b`, one request alone takes 1.5 to 4 s depending on how much of the model its integrated GPU took): the load test **2 to 3.5 minutes**, the whole notebook **2.5 to 4 minutes** across four runs. On a Colab **T4** a reply takes under a second, so expect about a minute for the load test plus the Ollama install (**56 s** measured in notebook 06's T4 run) and the model pull (**28 s**). The Colab T4 figure itself is not yet measured (`docs/timing_log.md`).
**Needs:** Google Colab on the free-tier **T4 GPU runtime** (*Runtime > Change runtime type*) — or the laptop that ran notebook 02, with Ollama installed and `llama3.2:1b` pulled (notebook 02's **PULL CELL**). **A Colab CPU runtime is refused:** notebook 02 measured 30 s a reply on two cores, and this lab sends 80 requests, so it cannot finish inside the budget there. **No API key.** Reads `data/eval/heldout_20.jsonl` (the 20 tickets used as prompts) and `scripts/concurrency_test.py` (the load driver).
**A correct result looks like:** the final cell prints `CONCURRENCY LAB DONE` with the chart, the table and three numbers for the sizing worksheet: the highest concurrency that stayed inside your latency target, the throughput at that level, and the requests per hour it implies. In the table, **p95 latency at 16 callers is several times the single-caller latency** and **throughput stops rising after the first level or two** — that is what "sequential backend" means, and it is the lesson. Zero failures at 64-token replies on a T4; on a slow laptop a few `timeout` failures at 16 callers are also a correct result (they are counted, not hidden).

> All data in this lab is synthetic. No real OQ material anywhere.

---
**The honesty rule of this lab, before any number appears.** Every figure this notebook prints comes from **this runtime**: a free Colab T4 or a laptop, a 1B model quantised to 8 bits, 64-token replies, one server process with Ollama's default of **one request at a time** (`OLLAMA_NUM_PARALLEL=1` on 0.12.10). None of that is what OQ would run, and **the numbers are not what OQ would see in production**: a production VM serves a bigger model through a serving engine built for concurrency (continuous batching, dozens of requests in flight), and its numbers would be different in every column. **The shape is the lesson — latency climbs with the queue, throughput hits a ceiling — and the shape is the same on a laptop, on a T4 and on that VM. The numbers are not transferable and the notebook never pretends they are.** The sizing worksheet asks you to carry three numbers out of here; carry them as *this machine's* numbers, and re-measure on the real one before anyone buys hardware.

**The plan.** Get the Ollama server answering (installed and started on Colab, reused on a laptop) → one request, timed, the baseline → **TODO 1:** choose the concurrency levels and run the load test (`scripts/concurrency_test.py`, the same script you can point at a VM or a vendor API later) → the chart → the table → **TODO 2:** set a latency target and read off the capacity → the three numbers.

**If the runtime disconnects:** reconnect and *Run all*. The load test's result is saved to your checkpoint folder (Google Drive on Colab) the moment it finishes, and the load-test cell loads it instead of running again. An interrupted load test is repeated from the start — it is three minutes, not thirty. On a fresh Colab runtime the Ollama install and the pull run again.


**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [1]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

Environment : local
Repo root   : C:\Users\utkar\oq-advanced-ai
Checkpoints : C:\Users\utkar\oq-advanced-ai\checkpoints\local


**Why this cell:** each notebook installs only what it needs, with exact pins matching `requirements.txt`. This one needs `requests` alone (the load driver uses it and the standard library, nothing else) — Colab preinstalls it at exactly this version, so on Colab the line is a no-op. Ollama is a server program, not a Python package; it gets its own cell below.

In [2]:
# Pinned installs — versions match requirements.txt. Colab only;
# local machines installed requirements.txt during setup.
if IN_COLAB:
    %pip install -q requests==2.32.4
print("Install cell done.")

Install cell done.


**Why this cell:** the settings that shape the test sit in one place, and each one is a lever a participant should know exists. `MODEL_NAME` is the Day 2 model (one pull serves notebooks 02, 03, 05 and 06), pushed into `OLLAMA_MODEL` so that the word `local` in `config/endpoints.py` — and in the load driver, which uses the same module — means this model. `REQUESTS_PER_LEVEL` decides how much each level is worth statistically: 16 requests make the p95 the second-slowest reply, which is honest and quick; 100 would be better and five times longer. `REPLY_CAP_TOKENS` keeps every request the same size, because a load test that lets reply lengths wander is measuring the model's mood, not the server. `TIMEOUT_SECONDS` is when a waiting caller gives up — the point at which a slow reply becomes a *lost* reply, which is the failure the room should see if the machine is slow enough.

The load driver is `scripts/concurrency_test.py`. It is imported here for its small helpers (the chart, the table, the capacity read-off) and run as a command for the test itself — the same command you could point at OQ's VM from a laptop.

In [3]:
import json
import time

import compare_utils     # run_command: run a script and show its output as it arrives
import inference_utils   # chat_timed: one request with the server's own clock
import ollama_utils
import utils

sys.path.insert(0, str(REPO_ROOT / "scripts"))
import concurrency_test  # the load driver: scripts/concurrency_test.py

MODEL_NAME = "llama3.2:1b"       # the Day 2 model; pulled in notebook 02
REQUESTS_PER_LEVEL = 16          # requests sent at each concurrency level
REPLY_CAP_TOKENS = 64            # every request asks for at most this many reply tokens
TIMEOUT_SECONDS = 60             # a caller that waits longer than this gives up: counted as a failure

# config/endpoints.py reads OLLAMA_MODEL to know which model "local" means. Tell it.
os.environ["OLLAMA_MODEL"] = MODEL_NAME

load_driver = REPO_ROOT / "scripts" / "concurrency_test.py"
server_log_path = REPO_ROOT / "eval_runs" / "ollama_server.log"   # `ollama serve` writes here if this notebook starts it
load_dir = CHECKPOINT_DIR / "concurrency"                          # the load test's files go here (Drive on Colab)
sitting_started = time.time()

print(f"model         : {MODEL_NAME}")
print(f"load per level: {REQUESTS_PER_LEVEL} requests, {REPLY_CAP_TOKENS}-token cap, {TIMEOUT_SECONDS} s timeout")
print(f"server        : {ollama_utils.server_url()}   (OLLAMA_BASE_URL)")
print(f"load driver   : {load_driver.relative_to(REPO_ROOT)}")
print(f"results       : {load_dir}")

model         : llama3.2:1b
load per level: 16 requests, 64-token cap, 60 s timeout
server        : http://localhost:11437   (OLLAMA_BASE_URL)
load driver   : scripts\concurrency_test.py
results       : C:\Users\utkar\oq-advanced-ai\checkpoints\local\concurrency


**Why this cell — the server, and how many requests it runs at once:** the same helper as notebook 02: an Ollama server that already answers on `OLLAMA_BASE_URL` is used as it is; otherwise one is started (and on Colab, first installed at the one pinned release, 0.12.10). The pull is a no-op when the model is there, and the warm-up loads it so the first measured request does not pay for the load.

One fact matters more here than in notebook 02: **how many requests this server processes at the same time.** Ollama calls it `OLLAMA_NUM_PARALLEL`. Its documentation still says the default "auto-selects 4 or 1 based on available memory"; the 0.12.10 source says **1** (`envconfig/config.go`), and a server this notebook starts inherits nothing else, so the answer here is 1: a queue with a single worker. The cell reads the number the server wrote into its own log when it loaded the model; if the server was already running before this notebook (a laptop's desktop app), the log is not ours and the cell says `unknown` — the load test tells you anyway, because latency starts doubling at exactly that many callers.

A Colab runtime without a GPU is refused here, on purpose: 80 requests at 30 s each is 40 minutes, and the budget is 15.

In [4]:
gpu = ollama_utils.gpu_name()
print(f"NVIDIA GPU: {gpu or 'none found'}")
if IN_COLAB and gpu is None:
    raise RuntimeError("This Colab runtime has no GPU: 80 requests at the CPU runtime's 30 s a reply is 40 minutes, "
                       "and the budget is 15. Runtime > Change runtime type > T4 GPU (free tier), then Run all.")

server = ollama_utils.ensure_server(IN_COLAB, log_dir=server_log_path.parent)
pull_status = ollama_utils.ensure_model(MODEL_NAME)
warm = ollama_utils.warm_up(MODEL_NAME)

if server["started_here"]:
    parallel_slots = concurrency_test.parallel_slots_from_log(server_log_path)
else:
    parallel_slots = None      # someone else's server: its log is not ours to read

print(f"Ollama {server['version']} at {server['url']}: "
      f"{'started by this notebook' if server['started_here'] else 'was already running - used as it is'}")
print(f"  {MODEL_NAME}: {pull_status}; warm-up {warm['seconds']} s; {warm['gpu_share']:.0%} of the model in GPU memory")
print(f"  requests processed at once (OLLAMA_NUM_PARALLEL): "
      f"{parallel_slots if parallel_slots is not None else 'unknown - the server was not started here'}")

NVIDIA GPU: none found


Ollama 0.12.10 at http://localhost:11437: started by this notebook
  llama3.2:1b: already there; warm-up 6.7 s; 100% of the model in GPU memory
  requests processed at once (OLLAMA_NUM_PARALLEL): 1


**Why this cell — milestone 1, the baseline:** one request, alone, with the server's own clock (the same helper as notebook 02's speed cell). Two numbers come out of it and both go on the worksheet: the **seconds for one reply** — the floor, which no latency target on this hardware can beat — and **tokens per second for one request at a time**, the figure that everything below divides up among callers. The prompt is a real held-out ticket with the house system prompt, the same prompts the load test uses, so the token counts are the ones OQ's tickets would produce.

In [5]:
prompts = concurrency_test.load_prompts(REPO_ROOT / "data" / "eval" / "heldout_20.jsonl")
print(f"{len(prompts)} ticket prompts; the load test rotates through them")

single = utils.load_json(CHECKPOINT_DIR, "03_single", default=None)
if single is None or single.get("model") != MODEL_NAME:
    single = inference_utils.chat_timed(MODEL_NAME, prompts[0],
                                        options={"temperature": 0.0, "num_predict": REPLY_CAP_TOKENS})
    single["model"] = MODEL_NAME
    single["measured_on"] = gpu or ("Colab CPU" if IN_COLAB else "local, no NVIDIA GPU")
    utils.save_json(CHECKPOINT_DIR, "03_single", single)
else:
    print(f"(from checkpoint - measured earlier on: {single['measured_on']})")

inference_utils.print_timing(single)

20 ticket prompts; the load test rotates through them


checkpoint saved: C:\Users\utkar\oq-advanced-ai\checkpoints\local\03_single.json
load model       :    0.17 s   (0 once it is already in memory)
read the prompt  :    0.69 s   (349 tokens)
write the reply  :    0.64 s   (64 tokens)
total            :    1.56 s
speed            :   100.0 tokens per second, one request at a time


**Why this cell — TODO 1, the load test:** now the same request arrives from several callers at once. The driver runs a *closed loop*: at each level it starts that many client threads, and every thread sends a request, waits for the reply, then sends the next, until the level's 16 requests are used up. That is the shape of N service-desk agents, or N integrations, each waiting on the model — not a flood of fire-and-forget requests. Per level it reports how many came back, the p50 / p95 / max latency of those that did, and throughput as requests per second and reply tokens per second.

Your job is to choose the levels. Start at 1 — that is the baseline again, measured the same way as everything else — and go up until the server is clearly past its limit. Watch the progress lines as it runs: the wall-clock time per level is itself the finding.

The result is saved to your checkpoint folder under a name that carries the model and the levels, so running this cell again loads it instead of sending 80 more requests. Change the levels and it runs again.

In [6]:
# ── TODO 1 ─────────────────────────────────────────────────────────
# The concurrency levels to test, lowest first: how many callers send at once.
# Start at 1 (the baseline) and go up until the server is clearly past its limit.
# Hint: doubling each step shows the shape in five levels; the driver's own default is [1, 2, 4, 8, 16].
LEVELS = [1, 2, 4, 8, 16]   # doubling: five levels, 80 requests, and the knee is visible whichever level it sits at
# ───────────────────────────────────────────────────────────────────

assert LEVELS is not ..., "TODO 1 is not filled in yet"

levels_text = ",".join(str(level) for level in LEVELS)
run_id = f"03_load_{MODEL_NAME.replace(':', '_')}_{levels_text.replace(',', '-')}"

load_test = utils.load_json(load_dir, f"{run_id}_summary", default=None)
if load_test is None:
    where = f"{'Colab' if IN_COLAB else 'local'}; {gpu or 'no NVIDIA GPU'}; parallel slots {parallel_slots}"
    command = [
        sys.executable, load_driver,
        "--endpoint", "local",
        "--levels", levels_text,
        "--requests", REQUESTS_PER_LEVEL,
        "--max-tokens", REPLY_CAP_TOKENS,
        "--timeout", TIMEOUT_SECONDS,
        "--out", load_dir,
        "--run-id", run_id,
        "--note", where,
    ]
    exit_code = compare_utils.run_command(command)
    assert exit_code == 0, "Nothing was measured - read the message above, fix it, run this cell again."
    load_test = utils.load_json(load_dir, f"{run_id}_summary")
else:
    print(f"(from checkpoint: measured {load_test['measured_at']} - {load_test['note']})")
    print(concurrency_test.render_table(load_test["levels"]))

total_requests = sum(row["requests"] for row in load_test["levels"])
total_failed = sum(row["failed"] for row in load_test["levels"])
print()
print(f"{len(load_test['levels'])} levels, {total_requests} requests, {total_failed} failed")

$ C:\Users\utkar\oq-advanced-ai\.venv\Scripts\python.exe C:\Users\utkar\oq-advanced-ai\scripts\concurrency_test.py --endpoint local --levels 1,2,4,8,16 --requests 16 --max-tokens 64 --timeout 60 --out C:\Users\utkar\oq-advanced-ai\checkpoints\local\concurrency --run-id 03_load_llama3.2_1b_1-2-4-8-16 --note "local; no NVIDIA GPU; parallel slots 1"


target : local (llama3.2:1b) at http://localhost:11437/v1
load   : levels [1, 2, 4, 8, 16], 16 requests per level, reply cap 64 tokens, timeout 60 s, 20 distinct ticket prompts
preflight: one request to http://localhost:11437/v1/chat/completions (model llama3.2:1b) ...


preflight: ok in 3.176 s, 64 reply tokens



concurrency   1: sending 16 requests ... done in 48.2 s  (p95 3.19 s, 0 failed)


concurrency   2: sending 16 requests ... done in 24.6 s  (p95 3.33 s, 0 failed)


concurrency   4: sending 16 requests ... done in 15.3 s  (p95 4.54 s, 0 failed)


concurrency   8: sending 16 requests ... done in 15.5 s  (p95 7.80 s, 0 failed)


concurrency  16: sending 16 requests ... done in 15.0 s  (p95 14.01 s, 0 failed)

concurrency  requests    ok  failed   p50 s   p95 s   max s   req/s  tokens/s
----------------------------------------------------------------------------------
          1        16    16       0    3.00    3.19    3.42    0.33      19.0
          2        16    16       0    2.95    3.33    3.71    0.65      37.3
          4        16    16       0    3.32    4.54    5.15    1.05      59.9
          8        16    16       0    6.77    7.80    8.46    1.03      59.0
         16        16    16       0    8.41   14.01   15.02    1.06      60.9

saved  : C:\Users\utkar\oq-advanced-ai\checkpoints\local\concurrency\03_load_llama3.2_1b_1-2-4-8-16_summary.json
         C:\Users\utkar\oq-advanced-ai\checkpoints\local\concurrency\03_load_llama3.2_1b_1-2-4-8-16_requests.jsonl

5 levels, 80 requests, 0 failed


**Why this cell — the chart, and what it is not:** one bar per level, the p95 latency: the reply an unlucky caller waited for. Read it with the honesty rule from the top of the notebook in front of you. **These bars are this runtime's — a free T4 or a laptop, a 1B model, 64-token replies, one request at a time — and they are not what OQ would see in production.** A production VM runs a larger model on a serving engine that batches requests, so its single-request latency, its knee and its ceiling would all sit somewhere else. What *does* carry over is the shape: past the number of requests the server handles at once, every extra caller is a place in a queue, and p95 grows roughly in step with the queue. Plain text on purpose: it reads the same on a projector, in a Windows console and in a saved file.

In [7]:
print(concurrency_test.text_chart(load_test["levels"]))

p95 latency (seconds) as concurrency rises

  1 at once |########### 3.19
  2 at once |############ 3.33
  4 at once |################ 4.54
  8 at once |############################ 7.80
 16 at once |################################################## 14.01

scale: '#' = 0.28 s;  the longest bar is 14.01 s


**Why this cell — the table, one line per level:** the chart's numbers plus the ones it hides. Read three columns against each other. **p50 versus p95:** at low concurrency they sit together; once there is a queue, p95 pulls away, because the unlucky caller is the one behind everybody. **req/s:** throughput. On a server that processes one request at a time it stops rising after the first level or two — more callers do not make the server faster, they make each other wait. **failed:** on a fast machine zero; on a slow laptop, a few `timeout` at 16 callers, and that is the honest result — the driver counts them, it does not average them in.

One thing in the table looks like a contradiction and is worth a minute: **throughput rises from one caller to two and again to four, on a server with one slot.** What is being overlapped is not the model. In the retained run its own clock put a request at 1.6 s (read the prompt, write 64 tokens) while a lone caller waited 3.0 s; the difference — request handling outside the model's own stages, plus the caller's turnaround — is what a queue hides. Once the queue is never empty, the server gets through a request every 0.95 s and stays there: the model still runs one request at a time, and the plateau from four callers on is its ceiling. (Where the gap goes exactly was not established, and whether it appears on a T4 is not yet measured — quote the plateau, not the gain.)

Two measured contrasts, so the room does not mistake the ceiling for a law of nature (build laptop, 2026-09-22, same script, same tickets, same 64-token cap). **Same server started with `OLLAMA_NUM_PARALLEL=4`:** throughput climbed from 0.25 to 0.9 requests a second before flattening at 8 callers, and p95 still went from 4.2 s to 17.6 s at 16 — batching moves the knee, it does not remove it. **The vendor API (`gpt-4o-mini`, 8 requests a level):** p95 2.8 s at one caller and 2.5 s at sixteen, throughput 0.45 to 3.25 requests a second and still rising — a fleet behind a load balancer, which is the thing OQ would be *buying* in the "buy" column of yesterday's matrix and *building* in the "host" column. Same reminder as the chart: the numbers are this machine's; the shape is the point.

In [8]:
print(concurrency_test.render_table(load_test["levels"]))
print()
first_level = load_test["levels"][0]
last_level = load_test["levels"][-1]
print(f"p95 latency  : {first_level['p95_seconds']} s at {first_level['concurrency']} caller"
      f" -> {last_level['p95_seconds']} s at {last_level['concurrency']} callers"
      f"  ({last_level['p95_seconds'] / first_level['p95_seconds']:.1f}x)")
print(f"throughput   : {first_level['requests_per_second']} req/s at {first_level['concurrency']}"
      f" -> {last_level['requests_per_second']} req/s at {last_level['concurrency']}"
      f"  ({last_level['requests_per_second'] / first_level['requests_per_second']:.1f}x)")
print(f"one request  : {single['tokens_per_second']} tokens/s alone;"
      f" the server's best level gave {max(row['reply_tokens_per_second'] for row in load_test['levels'])} tokens/s in total")

concurrency  requests    ok  failed   p50 s   p95 s   max s   req/s  tokens/s
----------------------------------------------------------------------------------
          1        16    16       0    3.00    3.19    3.42    0.33      19.0
          2        16    16       0    2.95    3.33    3.71    0.65      37.3
          4        16    16       0    3.32    4.54    5.15    1.05      59.9
          8        16    16       0    6.77    7.80    8.46    1.03      59.0
         16        16    16       0    8.41   14.01   15.02    1.06      60.9

p95 latency  : 3.19 s at 1 caller -> 14.01 s at 16 callers  (4.4x)
throughput   : 0.33 req/s at 1 -> 1.06 req/s at 16  (3.2x)
one request  : 100.0 tokens/s alone; the server's best level gave 60.9 tokens/s in total


**Why this cell — TODO 2, the capacity read-off:** a load test is not a result until somebody says what "too slow" means. Set a latency target: the slowest reply (p95) you would accept for a ticket-triage call that a service-desk tool is waiting on. The cell then finds the **highest concurrency level that stayed inside the target with no failures**, and turns it into the three numbers the sizing worksheet starts from: that concurrency, the throughput at it, and the requests per hour it implies. Set the target below the single-request time and the honest answer is "none" — this hardware cannot meet it even for one caller — and the cell says so instead of inventing a capacity.

Two hints for the number. A *person* waiting on a screen notices about two seconds. An *integration* — a ticketing system calling the model on every new ticket — usually has a timeout between 10 and 30 s, and cares about the p95, not the average. The load test's own timeout (60 s) is the outer wall.

In [9]:
# ── TODO 2 ─────────────────────────────────────────────────────────
# The latency target: the slowest reply (p95, in seconds) you would accept for one ticket-triage call.
# Hint: a person notices 2 s; an integration usually times out between 10 and 30 s; the single-request
# time above is the floor - a target below it cannot be met on this hardware by even one caller.
LATENCY_TARGET_SECONDS = 10   # an integration's timeout, not a person's patience: triage runs in the background
# ───────────────────────────────────────────────────────────────────

assert LATENCY_TARGET_SECONDS is not ..., "TODO 2 is not filled in yet"

inside = concurrency_test.highest_level_within(load_test["levels"], LATENCY_TARGET_SECONDS)

capacity = {
    "model": MODEL_NAME,
    "measured_on": load_test["note"],
    "latency_target_seconds": LATENCY_TARGET_SECONDS,
    "single_request_seconds": single["total_seconds"],
    "single_request_tokens_per_second": single["tokens_per_second"],
    "max_concurrency_inside_target": inside["concurrency"] if inside else 0,
    "p95_at_that_level": inside["p95_seconds"] if inside else None,
    "requests_per_second_at_that_level": inside["requests_per_second"] if inside else 0,
    "requests_per_hour": round(inside["requests_per_second"] * 3600) if inside else 0,
}
utils.save_json(CHECKPOINT_DIR, "03_capacity", capacity)

print(f"latency target      : p95 <= {LATENCY_TARGET_SECONDS} s")
if inside is None:
    print(f"capacity            : NONE - even 1 caller at a time got a p95 of {load_test['levels'][0]['p95_seconds']} s")
    print("                      on this hardware. The target needs a faster machine, a smaller reply, or a looser number.")
else:
    print(f"callers inside it   : {inside['concurrency']} at once (p95 {inside['p95_seconds']} s, {inside['failed']} failed)")
    print(f"throughput there    : {inside['requests_per_second']} requests/s = {capacity['requests_per_hour']} requests/hour")
    over = [row for row in load_test["levels"] if row["concurrency"] > inside["concurrency"]]
    if over:
        print(f"first level outside : {over[0]['concurrency']} callers (p95 {over[0]['p95_seconds']} s, {over[0]['failed']} failed)")
    else:
        print(f"first level outside : not reached - every level tested stayed inside; test higher levels to find it")
print()
print("-> Sizing worksheet, section A: these three numbers, for THIS machine, with the note beside them.")

checkpoint saved: C:\Users\utkar\oq-advanced-ai\checkpoints\local\03_capacity.json
latency target      : p95 <= 10 s
callers inside it   : 8 at once (p95 7.8 s, 0 failed)
throughput there    : 1.03 requests/s = 3708 requests/hour
first level outside : 16 callers (p95 14.01 s, 0 failed)

-> Sizing worksheet, section A: these three numbers, for THIS machine, with the note beside them.


**Why this cell:** the declared result in one block, so "done" can be checked from across the room, and the three worksheet numbers printed once more with the caveat attached to them, because they will be copied by hand and the caveat must travel with them. The last line is the hand-off to the talk: what a serving engine built for concurrency changes, and what it does not.

In [10]:
sitting_minutes = (time.time() - sitting_started) / 60

first_level = load_test["levels"][0]
last_level = load_test["levels"][-1]
print("CONCURRENCY LAB DONE")
print(f"  server         : Ollama {server['version']} at {server['url']}; {MODEL_NAME}; "
      f"{parallel_slots if parallel_slots is not None else 'unknown'} request(s) at once")
print(f"  hardware       : {gpu or 'no NVIDIA GPU'}; {warm['gpu_share']:.0%} of the model in GPU memory"
      f" ({'Colab' if IN_COLAB else 'local'})")
print(f"  one request    : {single['total_seconds']} s, {single['tokens_per_second']} tokens/s")
print(f"  load test      : {len(load_test['levels'])} levels ({levels_text}), {total_requests} requests, {total_failed} failed")
print(f"  p95 latency    : {first_level['p95_seconds']} s at {first_level['concurrency']} caller"
      f" -> {last_level['p95_seconds']} s at {last_level['concurrency']}")
print(f"  throughput     : {first_level['requests_per_second']} req/s at {first_level['concurrency']}"
      f" -> {last_level['requests_per_second']} req/s at {last_level['concurrency']}")
print(f"  for the sheet  : target p95 <= {LATENCY_TARGET_SECONDS} s -> {capacity['max_concurrency_inside_target']} callers,"
      f" {capacity['requests_per_second_at_that_level']} req/s, {capacity['requests_per_hour']} requests/hour")
print(f"                   (THIS machine's numbers - not OQ's production numbers; re-measure on the real server)")
print(f"  files          : {load_dir}")
print(f"  this sitting   : {sitting_minutes:.1f} minutes")
print()
print("Next, the talk: what a serving engine built for concurrency changes (the knee moves, the ceiling rises)")
print("and what it does not (there is still a knee, and still a ceiling).")

CONCURRENCY LAB DONE
  server         : Ollama 0.12.10 at http://localhost:11437; llama3.2:1b; 1 request(s) at once
  hardware       : no NVIDIA GPU; 100% of the model in GPU memory (local)
  one request    : 1.56 s, 100.0 tokens/s
  load test      : 5 levels (1,2,4,8,16), 80 requests, 0 failed
  p95 latency    : 3.19 s at 1 caller -> 14.01 s at 16
  throughput     : 0.33 req/s at 1 -> 1.06 req/s at 16
  for the sheet  : target p95 <= 10 s -> 8 callers, 1.03 req/s, 3708 requests/hour
                   (THIS machine's numbers - not OQ's production numbers; re-measure on the real server)
  files          : C:\Users\utkar\oq-advanced-ai\checkpoints\local\concurrency
  this sitting   : 2.4 minutes

Next, the talk: what a serving engine built for concurrency changes (the knee moves, the ceiling rises)
and what it does not (there is still a knee, and still a ceiling).
